# Module 3 • Classical Natural Language Processing

# Lesson 19 • Sequence Labeling and Named Entity Recognition Foundations

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 100–130 minutes

---

## Scope

This lesson introduces sequence labeling and Named Entity Recognition. It
covers token-level labels, BIO and BILOU tagging, entity spans, annotation
consistency, feature extraction, rule-based and classical baselines,
evaluation, error analysis, and Arabic NER considerations.

## Learning Objectives

After completing this lesson, the learner should be able to:

- define sequence labeling and Named Entity Recognition;
- distinguish token classification from document classification;
- explain entity types and span boundaries;
- encode entities using BIO and BILOU schemes;
- validate BIO sequences;
- convert between token labels and entity spans;
- build a rule-based NER baseline;
- construct token-level contextual features;
- train an independent-token Logistic Regression baseline;
- explain why sequence models such as HMMs and CRFs are useful;
- compare token-level and entity-level evaluation;
- calculate exact-span precision, recall, and F1;
- identify common NER errors;
- discuss Arabic tokenization, morphology, and orthographic challenges.

## Table of Contents

1. What Is Sequence Labeling?
2. What Is Named Entity Recognition?
3. Entity Types
4. Span Boundaries
5. BIO Tagging
6. BILOU Tagging
7. Validating Tag Sequences
8. Converting BIO Tags to Spans
9. Example Annotated Dataset
10. Rule-Based Baseline
11. Token-Level Feature Engineering
12. Independent-Token Classification
13. Why Sequence Models Matter
14. Hidden Markov Models
15. Conditional Random Fields
16. Token-Level Evaluation
17. Entity-Level Evaluation
18. Confusion Analysis
19. Boundary and Type Errors
20. Class Imbalance
21. Data Splitting and Leakage
22. Arabic NER Considerations
23. Reproducibility and Annotation Quality
24. Knowledge Check
25. Exercises
26. Summary and Next Lesson

# 1. What Is Sequence Labeling?

**Sequence labeling** assigns a label to every item in an ordered sequence.

For text, the items are usually tokens:

```text
Mona     visited     Cairo
LABEL    LABEL       LABEL
```

Common sequence-labeling tasks include:

- Named Entity Recognition;
- part-of-speech tagging;
- chunking;
- slot filling;
- semantic-role labeling;
- morphological tagging.

Sequence labeling differs from document classification because the model
predicts one label per token rather than one label for the entire document.

In [ ]:
import pandas as pd

task_comparison = pd.DataFrame(
    [
        ("Sentiment classification", "document", "positive"),
        ("Topic classification", "document", "technology"),
        ("Named Entity Recognition", "token", "B-PERSON"),
        ("Part-of-speech tagging", "token", "NOUN"),
        ("Slot filling", "token", "B-DESTINATION"),
    ],
    columns=["Task", "Prediction unit", "Example label"],
)

task_comparison

# 2. What Is Named Entity Recognition?

**Named Entity Recognition (NER)** identifies text spans referring to named or
structured entities and assigns them types.

Example:

```text
Mona visited Cairo on Monday.
```

Possible entities:

- `Mona` → PERSON;
- `Cairo` → LOCATION;
- `Monday` → DATE.

In [ ]:
example_entities = pd.DataFrame(
    [
        ("Mona", "PERSON"),
        ("Cairo", "LOCATION"),
        ("Monday", "DATE"),
    ],
    columns=["Entity text", "Entity type"],
)

example_entities

NER is used in:

- information extraction;
- search;
- question answering;
- document analytics;
- knowledge-graph construction;
- de-identification;
- customer-support automation.

# 3. Entity Types

Common entity types include:

| Type | Example |
|---|---|
| PERSON | Mona Ahmed |
| ORGANIZATION | OpenAI |
| LOCATION | Cairo |
| DATE | July 27, 2026 |
| TIME | 3:30 PM |
| MONEY | $500 |
| PRODUCT | Model X |
| EVENT | World Cup |

Entity inventories are task-specific. A medical dataset may contain DISEASE,
DRUG, and DOSAGE, while a legal dataset may contain COURT, CASE_NUMBER, and
STATUTE.

# 4. Span Boundaries

NER requires two decisions:

1. **Boundary:** which tokens belong to the entity?
2. **Type:** which entity category applies?

Example:

```text
New York University
```

Possible analyses:

- one ORGANIZATION span;
- `New York` as LOCATION plus `University` outside;
- a boundary error if only part of the organization is selected.

In [ ]:
boundary_examples = pd.DataFrame(
    [
        ("New York University", "ORGANIZATION", "full span"),
        ("New York", "LOCATION", "possible partial boundary"),
        ("University", "ORGANIZATION", "incomplete boundary"),
    ],
    columns=["Selected text", "Assigned type", "Observation"],
)

boundary_examples

Annotation guidelines must define whether titles, punctuation, legal forms, and
nested names belong inside entity spans.

# 5. BIO Tagging

The **BIO** scheme uses:

- `B-TYPE`: beginning of an entity;
- `I-TYPE`: inside the same entity;
- `O`: outside any entity.

Example:

```text
Mona       Ahmed      visited    New        York
B-PERSON   I-PERSON   O          B-LOCATION I-LOCATION
```

In [ ]:
bio_example = pd.DataFrame(
    [
        ("Mona", "B-PERSON"),
        ("Ahmed", "I-PERSON"),
        ("visited", "O"),
        ("New", "B-LOCATION"),
        ("York", "I-LOCATION"),
    ],
    columns=["Token", "BIO label"],
)

bio_example

A single-token entity receives `B-TYPE` in BIO.

# 6. BILOU Tagging

The **BILOU** scheme adds more boundary detail:

- `B`: beginning;
- `I`: inside;
- `L`: last;
- `U`: unit-length entity;
- `O`: outside.

Example:

```text
Mona       Ahmed      visited    Cairo
B-PERSON   L-PERSON   O          U-LOCATION
```

In [ ]:
bilou_example = pd.DataFrame(
    [
        ("Mona", "B-PERSON"),
        ("Ahmed", "L-PERSON"),
        ("visited", "O"),
        ("Cairo", "U-LOCATION"),
    ],
    columns=["Token", "BILOU label"],
)

bilou_example

BILOU makes single-token and final-token boundaries explicit but increases the
number of labels.

# 7. Validating Tag Sequences

A BIO sequence can be structurally invalid.

Invalid example:

```text
I-PERSON O
```

An `I-PERSON` tag should normally follow `B-PERSON` or `I-PERSON`.

In [ ]:
def validate_bio(labels: list[str]) -> list[str]:
    errors = []
    previous = "O"

    for index, label in enumerate(labels):
        if label == "O":
            previous = label
            continue

        if "-" not in label:
            errors.append(
                f"Token {index}: malformed label {label!r}"
            )
            previous = label
            continue

        prefix, entity_type = label.split("-", 1)

        if prefix not in {"B", "I"}:
            errors.append(
                f"Token {index}: unsupported prefix {prefix!r}"
            )

        if prefix == "I":
            valid_previous = {
                f"B-{entity_type}",
                f"I-{entity_type}",
            }

            if previous not in valid_previous:
                errors.append(
                    f"Token {index}: {label} cannot follow {previous}"
                )

        previous = label

    return errors


print(validate_bio(["B-PERSON", "I-PERSON", "O"]))
print(validate_bio(["I-PERSON", "O"]))

Automatic validation catches formatting errors before model training and
evaluation.

# 8. Converting BIO Tags to Spans

Entity-level evaluation requires converting token labels into spans.

In [ ]:
def bio_to_spans(
    tokens: list[str],
    labels: list[str],
) -> list[dict]:
    if len(tokens) != len(labels):
        raise ValueError("tokens and labels must have equal length")

    spans = []
    start = None
    current_type = None

    for index, label in enumerate(labels + ["O"]):
        if label == "O":
            if start is not None:
                spans.append(
                    {
                        "start_token": start,
                        "end_token": index,
                        "type": current_type,
                        "text": " ".join(tokens[start:index]),
                    }
                )
                start = None
                current_type = None
            continue

        prefix, entity_type = label.split("-", 1)

        if prefix == "B":
            if start is not None:
                spans.append(
                    {
                        "start_token": start,
                        "end_token": index,
                        "type": current_type,
                        "text": " ".join(tokens[start:index]),
                    }
                )

            start = index
            current_type = entity_type

        elif prefix == "I":
            if start is None or current_type != entity_type:
                if start is not None:
                    spans.append(
                        {
                            "start_token": start,
                            "end_token": index,
                            "type": current_type,
                            "text": " ".join(tokens[start:index]),
                        }
                    )

                start = index
                current_type = entity_type

    return spans


tokens = ["Mona", "Ahmed", "visited", "New", "York"]
labels = ["B-PERSON", "I-PERSON", "O", "B-LOCATION", "I-LOCATION"]

pd.DataFrame(bio_to_spans(tokens, labels))

The function repairs invalid `I` transitions by treating them as new spans.
Formal evaluation scripts may instead reject invalid sequences.

# 9. Example Annotated Dataset

Each sentence contains tokens and BIO labels.

In [ ]:
annotated_sentences = [
    (
        ["Mona", "works", "at", "OpenAI", "in", "Cairo"],
        ["B-PERSON", "O", "O", "B-ORGANIZATION", "O", "B-LOCATION"],
    ),
    (
        ["Ali", "visited", "Alexandria", "on", "Monday"],
        ["B-PERSON", "O", "B-LOCATION", "O", "B-DATE"],
    ),
    (
        ["Sara", "joined", "Microsoft", "last", "year"],
        ["B-PERSON", "O", "B-ORGANIZATION", "B-DATE", "I-DATE"],
    ),
    (
        ["Google", "opened", "an", "office", "in", "London"],
        ["B-ORGANIZATION", "O", "O", "O", "O", "B-LOCATION"],
    ),
    (
        ["Omar", "met", "Nadia", "in", "Dubai"],
        ["B-PERSON", "O", "B-PERSON", "O", "B-LOCATION"],
    ),
    (
        ["Amazon", "hired", "Lina", "in", "2025"],
        ["B-ORGANIZATION", "O", "B-PERSON", "O", "B-DATE"],
    ),
    (
        ["Youssef", "traveled", "to", "Paris"],
        ["B-PERSON", "O", "O", "B-LOCATION"],
    ),
    (
        ["IBM", "organized", "a", "conference", "in", "Berlin"],
        ["B-ORGANIZATION", "O", "O", "O", "O", "B-LOCATION"],
    ),
    (
        ["Eman", "presented", "the", "paper", "on", "Tuesday"],
        ["B-PERSON", "O", "O", "O", "O", "B-DATE"],
    ),
    (
        ["Meta", "invited", "Hassan", "to", "Rome"],
        ["B-ORGANIZATION", "O", "B-PERSON", "O", "B-LOCATION"],
    ),
]

len(annotated_sentences)

In [ ]:
rows = []

for sentence_id, (tokens, labels) in enumerate(annotated_sentences):
    for token_id, (token, label) in enumerate(zip(tokens, labels)):
        rows.append(
            {
                "sentence_id": sentence_id,
                "token_id": token_id,
                "token": token,
                "label": label,
            }
        )

token_data = pd.DataFrame(rows)
token_data.head(12)

Sentences, not individual tokens, should be split between training and test
partitions.

# 10. Rule-Based Baseline

A simple baseline can use dictionaries and surface patterns.

In [ ]:
PERSON_NAMES = {
    "Mona", "Ali", "Sara", "Omar", "Nadia",
    "Lina", "Youssef", "Eman", "Hassan",
}

ORGANIZATIONS = {
    "OpenAI", "Microsoft", "Google",
    "Amazon", "IBM", "Meta",
}

LOCATIONS = {
    "Cairo", "Alexandria", "London",
    "Dubai", "Paris", "Berlin", "Rome",
}

DATE_WORDS = {
    "Monday", "Tuesday", "2025",
}


def rule_based_ner(tokens: list[str]) -> list[str]:
    labels = []

    for token in tokens:
        if token in PERSON_NAMES:
            labels.append("B-PERSON")
        elif token in ORGANIZATIONS:
            labels.append("B-ORGANIZATION")
        elif token in LOCATIONS:
            labels.append("B-LOCATION")
        elif token in DATE_WORDS:
            labels.append("B-DATE")
        else:
            labels.append("O")

    return labels

In [ ]:
baseline_tokens = ["Mona", "visited", "Cairo", "on", "Tuesday"]
baseline_labels = rule_based_ner(baseline_tokens)

pd.DataFrame(
    {
        "token": baseline_tokens,
        "predicted_label": baseline_labels,
    }
)

Dictionary baselines are transparent but fail on unseen names, ambiguity, and
multi-token entities.

# 11. Token-Level Feature Engineering

Classical NER features may include:

- lowercase form;
- prefixes and suffixes;
- capitalization;
- token shape;
- digits;
- neighboring words;
- part-of-speech tags;
- lexicon membership;
- sentence position.

In [ ]:
def token_shape(token: str) -> str:
    result = []

    for character in token:
        if character.isupper():
            symbol = "X"
        elif character.islower():
            symbol = "x"
        elif character.isdigit():
            symbol = "d"
        else:
            symbol = character

        if not result or result[-1] != symbol:
            result.append(symbol)

    return "".join(result)


for token in ["Mona", "IBM", "2025", "OpenAI", "Cairo"]:
    print(f"{token:<8} -> {token_shape(token)}")

In [ ]:
def token_features(
    tokens: list[str],
    index: int,
) -> dict:
    token = tokens[index]

    features = {
        "token.lower": token.lower(),
        "token.prefix1": token[:1].lower(),
        "token.prefix2": token[:2].lower(),
        "token.suffix1": token[-1:].lower(),
        "token.suffix2": token[-2:].lower(),
        "token.is_title": token.istitle(),
        "token.is_upper": token.isupper(),
        "token.is_digit": token.isdigit(),
        "token.shape": token_shape(token),
        "position.is_first": index == 0,
        "position.is_last": index == len(tokens) - 1,
    }

    if index > 0:
        previous = tokens[index - 1]
        features.update(
            {
                "prev.lower": previous.lower(),
                "prev.is_title": previous.istitle(),
            }
        )
    else:
        features["BOS"] = True

    if index < len(tokens) - 1:
        following = tokens[index + 1]
        features.update(
            {
                "next.lower": following.lower(),
                "next.is_title": following.istitle(),
            }
        )
    else:
        features["EOS"] = True

    return features


token_features(
    ["Mona", "works", "at", "OpenAI"],
    0,
)

Context features help distinguish capitalized sentence-initial words from
names, but they do not enforce valid label transitions.

# 12. Independent-Token Classification

An independent-token classifier predicts each token separately using token and
local-context features.

This is a useful baseline, but it is not a true sequence model.

In [ ]:
training_sentence_ids = set(range(8))
test_sentence_ids = set(range(8, 10))

X_train_features = []
y_train_labels = []
X_test_features = []
y_test_labels = []
test_token_records = []

for sentence_id, (tokens, labels) in enumerate(annotated_sentences):
    for token_index, label in enumerate(labels):
        features = token_features(tokens, token_index)

        if sentence_id in training_sentence_ids:
            X_train_features.append(features)
            y_train_labels.append(label)
        else:
            X_test_features.append(features)
            y_test_labels.append(label)
            test_token_records.append(
                {
                    "sentence_id": sentence_id,
                    "token_id": token_index,
                    "token": tokens[token_index],
                }
            )

print("Training tokens:", len(X_train_features))
print("Test tokens:", len(X_test_features))

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

token_classifier = Pipeline(
    [
        (
            "vectorizer",
            DictVectorizer(
                sparse=True,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

token_classifier.fit(
    X_train_features,
    y_train_labels,
)

test_predictions = token_classifier.predict(
    X_test_features
)

test_predictions

In [ ]:
prediction_frame = pd.DataFrame(test_token_records)
prediction_frame["actual"] = y_test_labels
prediction_frame["predicted"] = test_predictions
prediction_frame["correct"] = (
    prediction_frame["actual"]
    == prediction_frame["predicted"]
)

prediction_frame

The small dataset is for workflow demonstration. Reliable NER requires a much
larger and more diverse annotated corpus.

# 13. Why Sequence Models Matter

Independent token classifiers can produce invalid outputs:

```text
O I-PERSON O
```

A sequence model considers dependencies among neighboring labels.

Useful dependencies include:

- `I-PERSON` often follows `B-PERSON`;
- an entity type tends to remain consistent inside a span;
- some label transitions are unlikely or invalid.

# 14. Hidden Markov Models

A **Hidden Markov Model (HMM)** is a generative sequence model.

It estimates:

- transition probabilities between labels;
- emission probabilities from labels to observed words.

Simplified factorization:

```text
P(labels, words)
=
transition probabilities
×
emission probabilities
```

HMM assumptions are restrictive:

- the current label depends mainly on the previous label;
- the current observation depends mainly on the current label.

Classical HMMs may struggle to incorporate many overlapping features.

# 15. Conditional Random Fields

A **Conditional Random Field (CRF)** is a discriminative sequence model.

CRFs can combine:

- word features;
- prefixes and suffixes;
- capitalization;
- neighboring context;
- transition preferences.

They predict the best label sequence jointly rather than independently.

CRFs were strong classical NER models before neural sequence models became
dominant. They remain valuable baselines and are interpretable at the feature
level.

# 16. Token-Level Evaluation

Token-level evaluation treats every token label as one classification
decision.

Metrics include:

- token accuracy;
- per-label precision;
- per-label recall;
- per-label F1;
- macro F1.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

token_accuracy = accuracy_score(
    y_test_labels,
    test_predictions,
)

print(f"Token accuracy: {token_accuracy:.3f}")
print()
print(
    classification_report(
        y_test_labels,
        test_predictions,
        zero_division=0,
    )
)

Token accuracy may be misleading because most tokens often receive `O`.

# 17. Entity-Level Evaluation

Exact entity-level evaluation requires both the correct span and the correct
type.

In [ ]:
def sentence_predictions_from_frame(
    frame: pd.DataFrame,
    label_column: str,
) -> dict[int, list[str]]:
    result = {}

    for sentence_id, group in frame.groupby("sentence_id"):
        ordered = group.sort_values("token_id")
        result[int(sentence_id)] = ordered[label_column].tolist()

    return result


actual_by_sentence = sentence_predictions_from_frame(
    prediction_frame,
    "actual",
)

predicted_by_sentence = sentence_predictions_from_frame(
    prediction_frame,
    "predicted",
)

actual_by_sentence, predicted_by_sentence

In [ ]:
def span_set(
    tokens: list[str],
    labels: list[str],
) -> set[tuple[int, int, str]]:
    return {
        (
            item["start_token"],
            item["end_token"],
            item["type"],
        )
        for item in bio_to_spans(tokens, labels)
    }


gold_entities = set()
predicted_entities = set()

for sentence_id in sorted(test_sentence_ids):
    tokens, _ = annotated_sentences[sentence_id]

    for span in span_set(
        tokens,
        actual_by_sentence[sentence_id],
    ):
        gold_entities.add((sentence_id, *span))

    for span in span_set(
        tokens,
        predicted_by_sentence[sentence_id],
    ):
        predicted_entities.add((sentence_id, *span))

print("Gold entities:", gold_entities)
print("Predicted entities:", predicted_entities)

In [ ]:
true_positive = len(
    gold_entities & predicted_entities
)

precision = (
    true_positive / len(predicted_entities)
    if predicted_entities
    else 0.0
)

recall = (
    true_positive / len(gold_entities)
    if gold_entities
    else 0.0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0
    else 0.0
)

print(f"Exact entity precision: {precision:.3f}")
print(f"Exact entity recall:    {recall:.3f}")
print(f"Exact entity F1:        {f1:.3f}")

Entity-level F1 is usually more informative than token accuracy for NER.

# 18. Confusion Analysis

A confusion matrix shows which labels are confused.

In [ ]:
from sklearn.metrics import confusion_matrix

ner_labels = sorted(
    set(y_test_labels) | set(test_predictions)
)

confusion = confusion_matrix(
    y_test_labels,
    test_predictions,
    labels=ner_labels,
)

pd.DataFrame(
    confusion,
    index=[f"actual_{label}" for label in ner_labels],
    columns=[f"predicted_{label}" for label in ner_labels],
)

Confusion analysis may reveal that the system finds entity boundaries but
assigns the wrong type.

# 19. Boundary and Type Errors

Common NER error categories include:

- false positive;
- false negative;
- correct boundary, wrong type;
- partial span;
- merged entities;
- split entity;
- invalid BIO transition;
- tokenization misalignment.

In [ ]:
ner_error_examples = pd.DataFrame(
    [
        ("New York University", "New York", "partial span"),
        ("OpenAI", "PERSON", "wrong type"),
        ("Mona Ahmed", "Mona / Ahmed", "split entity"),
        ("Cairo and Giza", "one LOCATION span", "merged entities"),
        ("I-PERSON after O", "invalid BIO", "transition error"),
    ],
    columns=["Gold or input", "Predicted output", "Error type"],
)

ner_error_examples

Boundary errors may arise from tokenization rather than the sequence model.

# 20. Class Imbalance

The `O` label usually dominates NER datasets.

Consequences include:

- high accuracy from predicting `O`;
- weak entity recall;
- poor rare-type performance;
- unstable minority-class metrics.

In [ ]:
label_distribution = (
    token_data["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="count")
)

label_distribution["percentage"] = (
    label_distribution["count"]
    / label_distribution["count"].sum()
    * 100
).round(1)

label_distribution

Use per-class metrics and exact entity F1 rather than accuracy alone.

# 21. Data Splitting and Leakage

Split data by sentence, document, or conversation—not by individual tokens.

Token-level splitting can place neighboring tokens from one sentence in both
training and test sets.

Other leakage risks include:

- duplicate sentences across splits;
- one document divided across splits;
- gazetteers built from the complete dataset;
- annotation notes included as features;
- test entities added to training dictionaries.

In [ ]:
leakage_examples = pd.DataFrame(
    [
        ("Random token split", "same sentence appears in train and test"),
        ("Duplicate news article", "near-identical context leaks"),
        ("Full-data name dictionary", "test entity vocabulary leaks"),
        ("Document split after sentence extraction", "document context leaks"),
    ],
    columns=["Unsafe choice", "Risk"],
)

leakage_examples

# 22. Arabic NER Considerations

Arabic NER must account for:

- attached conjunctions and prepositions;
- the definite article;
- rich morphology;
- omitted short vowels;
- spelling variation;
- weak capitalization cues;
- Modern Standard Arabic and dialects;
- transliteration and code-switching;
- tokenization standards.

Example:

```text
وبالقاهرة
```

may contain:

```text
و + ب + القاهرة
and + in + Cairo
```

A whitespace tokenizer treats the entire form as one token.

In [ ]:
arabic_ner_examples = pd.DataFrame(
    [
        ("زار محمد القاهرة", "محمد", "PERSON"),
        ("زار محمد القاهرة", "القاهرة", "LOCATION"),
        ("عملت سارة في مايكروسوفت", "سارة", "PERSON"),
        ("عملت سارة في مايكروسوفت", "مايكروسوفت", "ORGANIZATION"),
        ("سافر علي إلى دبي", "علي", "PERSON"),
        ("سافر علي إلى دبي", "دبي", "LOCATION"),
    ],
    columns=["Sentence", "Entity", "Type"],
)

arabic_ner_examples

Character n-grams, morphological features, lexicons, and pretrained language
models can help, but evaluation should cover dialect and domain variation.

## 22.1 Arabic BIO Example

In [ ]:
arabic_bio = pd.DataFrame(
    [
        ("زار", "O"),
        ("محمد", "B-PERSON"),
        ("مدينة", "O"),
        ("نيو", "B-LOCATION"),
        ("يورك", "I-LOCATION"),
    ],
    columns=["Token", "BIO label"],
)

arabic_bio

Entity boundaries depend on the selected Arabic segmentation scheme.

# 23. Reproducibility and Annotation Quality

Record:

- entity inventory;
- annotation guidelines;
- tokenization standard;
- BIO or BILOU scheme;
- train-validation-test split;
- model features;
- random seed;
- evaluation script;
- invalid-sequence handling;
- software versions.

In [ ]:
import sklearn

experiment_metadata = pd.Series(
    {
        "sentences": len(annotated_sentences),
        "tagging_scheme": "BIO",
        "classifier": "Logistic Regression token baseline",
        "sentence_level_split": True,
        "random_state": 42,
        "scikit-learn_version": sklearn.__version__,
    },
    name="NER experiment",
)

experiment_metadata

Annotation quality should be assessed through:

- pilot annotation;
- agreement measurement;
- boundary-error review;
- adjudication;
- automated BIO validation;
- versioned guidelines.

# 24. Knowledge Check

1. What is sequence labeling?
2. How does NER differ from document classification?
3. What are entity boundaries?
4. What do B, I, and O represent?
5. How does BILOU differ from BIO?
6. What makes a BIO sequence invalid?
7. Why convert token labels to spans?
8. What are common classical NER features?
9. Why is an independent-token classifier not a true sequence model?
10. What does an HMM model?
11. Why are CRFs useful for NER?
12. Why can token accuracy be misleading?
13. What is exact entity-level F1?
14. Why should data be split by sentence or document?
15. Which Arabic characteristics complicate NER?

# 25. Exercises

## Exercise 1 — BIO Annotation

Annotate ten sentences containing PERSON, ORGANIZATION, LOCATION, and DATE
entities.

## Exercise 2 — BIO Validation

Extend the validator to repair or reject malformed sequences.

## Exercise 3 — Span Conversion

Implement conversion from spans back to BIO labels.

## Exercise 4 — Rule-Based NER

Build a dictionary and pattern baseline for names, dates, and organizations.

## Exercise 5 — Token Features

Add prefixes, suffixes, word shape, neighboring words, and lexicon membership.

## Exercise 6 — Evaluation

Compare token accuracy, token macro F1, and exact entity F1.

## Exercise 7 — Error Analysis

Categorize false positives, false negatives, boundary errors, and type errors.

## Exercise 8 — Arabic NER

Create an Arabic BIO dataset and document the tokenization policy.

## Challenge Exercises

1. Train a CRF using an appropriate sequence-labeling library.
2. Compare BIO and BILOU tagging.
3. Add gazetteer features without leaking test entities.
4. Evaluate performance by entity type and entity length.
5. Build a complete NER annotation and evaluation report.

# 26. Summary and Next Lesson

In this lesson:

- sequence labeling assigned one label per token;
- NER identified entity spans and types;
- BIO and BILOU encoded entity boundaries;
- validation detected malformed tag sequences;
- span conversion enabled exact entity-level evaluation;
- rule-based NER provided a transparent baseline;
- contextual token features supported classical classification;
- independent-token models ignored transition constraints;
- HMMs modeled emissions and transitions generatively;
- CRFs modeled label sequences discriminatively;
- token accuracy was distinguished from entity-level F1;
- boundary, type, and transition errors required separate analysis;
- sentence-level splitting prevented token-context leakage;
- Arabic NER required explicit tokenization and morphology policies.

## Next Lesson

**Lesson 20: Classical NLP Capstone — Building and Evaluating an End-to-End
Text Analytics Pipeline** integrates preprocessing, vectorization,
classification, retrieval, topic modeling, and evaluation into one reproducible
project.

# References

- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- Lafferty, J., McCallum, A., & Pereira, F. *Conditional Random Fields*.
- classical Hidden Markov Model and sequence-labeling literature.
- CoNLL Named Entity Recognition task documentation.
- Arabic Named Entity Recognition literature.